In [0]:
%sql
SELECT *
FROM silver.sf_salaries.silver AS t1
WHERE t1.DateJob < 2012


In [0]:
%sql
SELECT t1.DateJob,
       count(*)
FROM silver.sf_salaries.silver  AS t1
GROUP BY ALL
ORDER BY t1.DateJob

In [0]:
%pip install databricks-feature-engineering

In [0]:
import datetime
from databricks.feature_engineering import FeatureEngineeringClient 
from databricks.feature_engineering import FeatureEngineeringClient
from dateutil.relativedelta import relativedelta
from tqdm import tqdm

In [0]:
fe = FeatureEngineeringClient()

def range_date(dt_start, dt_end):
    dates = []
    dt_start = datetime.datetime.strptime(dt_start, '%Y')
    dt_end = datetime.datetime.strptime(dt_end, '%Y')
    while dt_start <= dt_end:
        # Armazena estritamente o ano puro (Ex: '2011')
        dates.append(dt_start.strftime('%Y'))
        # Incrementa exatamente 1 ano a cada passo do loop
        dt_start += relativedelta(years=1)
    
    return dates


In [0]:
range_date(dt_start='2011', dt_end='2014')

In [0]:
catalog = 'features_store'
database = 'sf_salaries'
table = 'features_salaries'
table_name = f'{catalog}.{database}.{table}' 
dt_start = '2011'
dt_end = '2014'
query_name = 'sf_salaries'
primary_keys = ['dt_ref', 'id_funcionario']
partition_by = 'dt_ref'


dates = range_date(dt_start=dt_start, dt_end=dt_end)
with open(f"./{query_name}.sql", "r") as file_open:
    query = file_open.read()

def tables_exists(spark, catalog, database, table):
    count = (spark.sql(f"SHOW TABLES FROM {catalog}.{database}")
                    .filter(f"tableName = '{table}'")
                    .count())     
    return count == 1

In [0]:
if not tables_exists(spark, catalog, database, table):
    
    df = spark.sql(query.format(dt_ref=dates.pop(0)))
    df = df.dropDuplicates(['dt_ref', 'id_funcionario'])    
    fe.create_table(
                    df=df,
                    name= table_name,
                    primary_keys= primary_keys,
                    partition_columns= partition_by
                        )

for d in tqdm(dates):
    
    df = spark.sql(query.format(dt_ref=d)) 
    # Atualizar os dados da tabela
    fe.write_table(df=df.dropDuplicates(['dt_ref', 'id_funcionario']), name=table_name, mode='merge')

In [0]:
%sql
SELECT t1.dt_ref,
       count(*)
FROM features_store.sf_salaries.features_salaries  AS t1
GROUP BY ALL
ORDER BY t1.dt_ref